<a href="https://colab.research.google.com/github/0301sdh/faceguard/blob/main/notebooks/ipadapter_faceid_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IP-Adapter FaceID 보호 효과 실험 (FaceGuard Week 7)

실제로 막고 싶은 위협("내 얼굴이 다른 장면·몸에 들어간 이미지")을 만드는 신원 유지 이미지 생성 도구로 평가 대상을 넓힌다.

**처음 한 번만:** Week 6과 같은 사진을 쓴다 (`MyDrive/faceguard/photos/person_a_1.jpg` ~ `person_a_4.jpg`).

**실행 순서:** 런타임을 T4 GPU로 설정 → `준비 1` 실행 (끝나면 런타임이 자동으로 다시 시작됨) → `준비 2` → `도구 함수` → 실험 셀

**결과 저장:** `MyDrive/faceguard/week7/<tag>/`. FaceNet 채점은 Week 6처럼 Mac에서 한다 (Colab에 facenet-pytorch를 설치하면 PyTorch 버전이 내려가므로).

출처: [IP-Adapter](https://github.com/tencent-ailab/IP-Adapter), [IP-Adapter-FaceID](https://huggingface.co/h94/IP-Adapter-FaceID) (비상업 연구 용도), [insightface](https://github.com/deepinsight/insightface) (모델은 비상업 연구 용도), [diffusers](https://github.com/huggingface/diffusers). 결과와 해석: `docs/weekly/week7.md`

## 준비

In [ ]:
# [준비 1] 패키지 설치. 끝나면 런타임이 자동으로 다시 시작된다.
# - diffusers: Stable Diffusion과 IP-Adapter를 실행하는 라이브러리
# - insightface: FaceID가 신원을 읽을 때 쓰는 얼굴 검출 + ArcFace
# insightface 모델(.onnx 파일)은 Colab에 이미 있는 onnxruntime(CPU용)으로 실행한다.
#   얼굴 몇 장을 검출·채점하는 데는 CPU로 충분하다. onnxruntime-gpu를 같이 깔면 CPU용이 우선되어 어차피 GPU를 못 씀
# Week 5~6 노트북은 SimSwap 때문에 옛 insightface(0.2.1)를 썼지만, 여기서는 FaceID 예시와 같은 최신 버전을 쓴다.
!pip install -q diffusers insightface
# 최신 diffusers는 torchao(양자화 도구)가 깔려 있으면 불러오는데, Colab에 있는 옛 torchao에는 필요한 기능(FqnToConfig)이 없어 오류가 남.
# 우리는 양자화를 쓰지 않으므로 지움 → diffusers가 torchao 없이 동작
!pip uninstall -q -y torchao

# 새로 설치한 패키지를 쓰려면 런타임을 다시 시작해야 한다. 스스로 종료하면 Colab이 자동으로 다시 시작한다.
import os
os.kill(os.getpid(), 9)

In [ ]:
# [준비 2] 다시 시작된 뒤 실행. Drive 연결 + 모델 불러오기 (처음에는 내려받느라 몇 분 걸림)
import os
import cv2
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')
PHOTO_DIR = '/content/drive/MyDrive/faceguard/photos'
OUT_DIR = '/content/drive/MyDrive/faceguard/week7'

# 1) insightface: FaceID가 신원을 읽는 '목격자'. buffalo_l 안의 검출 모델과 ArcFace만 불러옴
from insightface.app import FaceAnalysis
app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition'],
                   providers=['CPUExecutionProvider'])
app.prepare(ctx_id=-1, det_size=(640, 640))   # ctx_id=-1: CPU로 실행

# 2) Stable Diffusion 1.5 + IP-Adapter FaceID. 설정은 FaceID 모델 카드 예시를 따름
#    - Realistic Vision: 사진처럼 사실적인 얼굴을 잘 그리도록 조정된 SD 1.5 모델 (VAE는 따로 받음)
#    - DDIM: 노이즈를 빼는 방식(스케줄러). 모델 카드와 같은 값
#    - float16: 메모리를 절반으로 줄여 T4(16GB)에서 여유 있게 돌림
from diffusers import StableDiffusionPipeline, DDIMScheduler, AutoencoderKL
scheduler = DDIMScheduler(num_train_timesteps=1000, beta_start=0.00085, beta_end=0.012,
                          beta_schedule='scaled_linear', clip_sample=False,
                          set_alpha_to_one=False, steps_offset=1)
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse', torch_dtype=torch.float16)
pipe = StableDiffusionPipeline.from_pretrained(
    'SG161222/Realistic_Vision_V4.0_noVAE', scheduler=scheduler, vae=vae,
    torch_dtype=torch.float16, safety_checker=None, feature_extractor=None).to('cuda')

# 3) UNet에 FaceID 통로(cross-attention)를 붙임. FaceID는 CLIP 이미지 인코더를 쓰지 않으므로 image_encoder_folder=None
pipe.load_ip_adapter('h94/IP-Adapter-FaceID', subfolder=None,
                     weight_name='ip-adapter-faceid_sd15.bin', image_encoder_folder=None)
# 얼굴 정보를 얼마나 세게 반영할지 (0~1). 공격자가 가장 닮게 만들려는 상황을 가정해 1.0
pipe.set_ip_adapter_scale(1.0)

# 실행 환경 기록 (week7.md에 남김)
import diffusers, insightface, onnxruntime
print('diffusers', diffusers.__version__, '/ insightface', insightface.__version__,
      '/ onnxruntime', onnxruntime.__version__, onnxruntime.get_device())

## 도구 함수
세션을 새로 열 때마다 실행한다. 함수를 정의만 하므로 바로 끝난다.

In [ ]:
# [도구 함수]
def add_margin(img_bgr, pad=112):
    # 얼굴만 꽉 찬 224×224 사진 둘레에 검은 여백 → 검출 모델이 얼굴을 찾기 쉽게 (Week 6과 같음)
    return cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))

def face_embed(img_bgr):
    # 사진 → insightface 얼굴 검출·정렬 → ArcFace → 길이 1인 512개 숫자. 얼굴을 못 찾으면 None
    # FaceID가 생성 모델에 넘기는 값과 똑같다 (내 사진에서 생성 모델로 가는 유일한 정보)
    faces = app.get(add_margin(img_bgr))
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))   # 여러 개면 가장 큰 얼굴
    return torch.from_numpy(face.normed_embedding)

NEGATIVE = 'monochrome, lowres, bad anatomy, worst quality, low quality, blurry'

def generate(emb, prompt, seed):
    # 얼굴 명세서(512개 숫자) + 프롬프트 → 이미지 (OpenCV BGR)
    # 모양 (1, 1, 1, 512): diffusers 문서의 FaceID 예시와 같은 모양
    pos = emb.view(1, 1, 1, 512)
    # classifier-free guidance: '얼굴 정보 있음'과 '없음(0)' 두 번 예측해서 그 차이를 키움 → 얼굴을 더 강하게 반영
    neg = torch.zeros_like(pos)
    id_embeds = torch.cat([neg, pos]).to('cuda', torch.float16)
    g = torch.Generator('cuda').manual_seed(seed)   # 같은 seed → 같은 출발 노이즈 → 비교 가능
    img = pipe(prompt=prompt, negative_prompt=NEGATIVE, ip_adapter_image_embeds=[id_embeds],
               num_inference_steps=30, width=512, height=512, generator=g).images[0]
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def cos(a, b):
    # 둘 다 길이 1이므로 내적 = 코사인 유사도
    return float((a * b).sum())

## 실험 기록

### 확인: 환경이 돌아가는가 (내 사진 1장 → 이미지 1장)
- 얼굴을 찾고, 이미지가 생성되고, Drive에 저장되면 성공
- 유사도 숫자는 참고만 한다. baseline은 다음 단계에서 사진 4장 × seed 여러 개로 잰다

In [ ]:
os.makedirs(f'{OUT_DIR}/check', exist_ok=True)
src = cv2.imread(f'{PHOTO_DIR}/person_a_1.jpg')
emb = face_embed(src)
print('얼굴 찾음' if emb is not None else '얼굴 못 찾음')

# 얼굴이 너무 작으면 채점할 때 검출이 안 되므로, 처음에는 상반신 장면으로
prompt = 'upper body photo of a person wearing a suit in an office'
out = generate(emb, prompt, seed=0)
cv2.imwrite(f'{OUT_DIR}/check/a_1_seed0.png', out)

out_emb = face_embed(out)
print('생성 결과 vs 내 사진 (insightface ArcFace):',
      '얼굴 못 찾음' if out_emb is None else round(cos(emb, out_emb), 3))

from google.colab.patches import cv2_imshow
cv2_imshow(np.hstack([cv2.resize(src, (512, 512)), out]))

### 1. 눈금 잡기: insightface ArcFace에서 본인과 타인은 몇인가
- 0.746이 높은지 판단하려면 이 모델의 기준선이 필요하다 (FaceNet·SimSwap ArcFace와는 눈금이 다름)
- 판정 기준은 Week 5와 같은 방식: 본인끼리 최소 = 실패선, 본인 최소와 타인의 중간 = 성공선
- **처음 한 번만:** Drive `photos/`에 타인 사진 `target_specific1.png`(Week 5~6의 SimSwap target, Mac `data/week5/`)를 올려 둔다

In [ ]:
# [눈금 잡기] 내 원본 사진 4장의 명세서 = 채점 기준 (Week 6과 같음)
from itertools import combinations
names = ['person_a_1', 'person_a_2', 'person_a_3', 'person_a_4']
mine = {n: face_embed(cv2.imread(f'{PHOTO_DIR}/{n}.jpg')) for n in names}
print('얼굴 못 찾은 사진:', [n for n in names if mine[n] is None] or '없음')

same = [cos(mine[a], mine[b]) for a, b in combinations(names, 2)]
print('본인끼리 (6쌍):', ' '.join(f'{s:.3f}' for s in same), f'| 최소 {min(same):.3f}')

fail_line, success_line = min(same), None
other_img = cv2.imread(f'{PHOTO_DIR}/target_specific1.png')   # Week 5~6과 같은 SimSwap target → FaceNet 채점과 같은 타인
if other_img is None:
    print('타인 사진이 없음: Drive photos/에 target_specific1.png를 올리고 다시 실행')
else:
    other = face_embed(other_img)
    others = [cos(mine[n], other) for n in names]
    success_line = (fail_line + np.mean(others)) / 2
    print('타인과 (4쌍):', ' '.join(f'{s:.3f}' for s in others), f'| 평균 {np.mean(others):.3f}')
    print(f'ArcFace 판정 기준: 실패 >= {fail_line:.3f} | 부분 {success_line:.3f} ~ {fail_line:.3f} | 성공 < {success_line:.3f}')

def judge(score):
    # 생성 결과가 나를 닮았으면 '실패'(보호 못 함), 타인 수준이면 '성공'
    if score is None or success_line is None:
        return '-'
    if score >= fail_line:
        return '실패'
    return '부분' if score >= success_line else '성공'

### 2. Baseline: 보호 안 한 얼굴로 생성하면 얼마나 나를 닮나
- **입력:** Week 6의 보호 안 한 얼굴 crop (`week6/robust_eps0/person_a_k_eps0.png`)
  - 다음 단계의 보호 사진(`eps5`)도 같은 224×224 정렬 얼굴이다. 입력 형식을 같게 해야 차이가 **노이즈 때문**이라고 말할 수 있다
- 사진 4장 × seed 3개 = 12장, 프롬프트 1개 고정 (T4에서 약 2분)
- **채점:** 생성 결과 vs 내 원본 사진 (Week 6과 같은 기준). FaceNet 채점은 Mac에서 `experiments/week7_facenet_eval.py baseline`
- 눈으로 닮았는지도 사진마다 한 줄씩 기록한다
- ⚠️ 얼굴 사진이 출력되므로, GitHub에 올리기 전에 출력을 지우거나 "저장할 때 코드 셀 출력 생략"을 켜 둔다

In [ ]:
# [생성 + 채점] 사진별·seed별로 생성해 Drive week7/<tag>/에 저장하고 ArcFace로 채점
PROMPT = 'upper body photo of a person wearing a suit in an office'
SEEDS = [0, 1, 2]

def run_gen(tag, inputs):
    # inputs: {이름: 입력 사진 경로}
    # baseline과 보호 사진에 같은 seed를 쓰면 출발 노이즈가 같아짐 → 입력(명세서)만 다른 공정한 비교
    out_dir = f'{OUT_DIR}/{tag}'
    os.makedirs(out_dir, exist_ok=True)
    scores = {}
    for name, path in inputs.items():
        emb = face_embed(cv2.imread(path))   # FaceID가 받는 명세서
        if emb is None:
            print(f'[{tag}] {name}: 입력에서 얼굴 못 찾음')
            continue
        row = []
        for seed in SEEDS:
            out = generate(emb, PROMPT, seed)
            cv2.imwrite(f'{out_dir}/{name}_seed{seed}.png', out)
            out_emb = face_embed(out)
            score = None if out_emb is None else cos(mine[name], out_emb)   # 채점 기준: 원본 내 사진
            scores[(name, seed)] = score
            row.append('-' if score is None else f'{score:.3f}')
        values = [scores[(name, s)] for s in SEEDS if scores[(name, s)] is not None]
        mean = np.mean(values) if values else None
        shown = '-' if mean is None else f'{mean:.3f}'
        print(f'[{tag}] {name} | seed별 {" ".join(row)} | 평균 {shown} ({judge(mean)})')
    all_values = [v for v in scores.values() if v is not None]
    print(f'[{tag}] 4장 평균 {np.mean(all_values):.3f} ({judge(np.mean(all_values))})')
    return scores

def show(tag, inputs):
    # 사진마다 [입력 | seed0 | seed1 | seed2]를 한 줄로 → 눈으로 닮았는지 확인
    from google.colab.patches import cv2_imshow
    for name, path in inputs.items():
        row = [cv2.resize(cv2.imread(path), (256, 256))]
        row += [cv2.resize(cv2.imread(f'{OUT_DIR}/{tag}/{name}_seed{s}.png'), (256, 256)) for s in SEEDS]
        print(name)
        cv2_imshow(np.hstack(row))

In [ ]:
CLEAN = '/content/drive/MyDrive/faceguard/week6/robust_eps0'
baseline_inputs = {n: f'{CLEAN}/{n}_eps0.png' for n in names}
baseline = run_gen('baseline', baseline_inputs)
show('baseline', baseline_inputs)

### 3. Pilot: Week 6 보호 사진을 넣으면 덜 닮나
- **입력:** Week 6 최종 설정(eps 5/255, EOT 변형 4개)의 보호 얼굴 (`week6/eps5_n4_seed0/person_a_k_eps5.png`)
- `PROMPT`, `SEEDS`는 baseline과 같다 → 출발 노이즈가 같아서 **입력의 노이즈만 다른** 비교
- 먼저 노이즈 seed0 하나로 방향을 보고, 효과에 따라 seed1·seed2를 돌릴지 정한다
  - 확실히 떨어짐 → SimSwap ArcFace로 만든 노이즈가 FaceID(insightface ArcFace)에도 전이됨
  - 거의 안 떨어짐 → 두 ArcFace가 달라서 생긴 전이 문제 → 함께 공격(ensemble) 검토
- FaceNet 채점: `experiments/week7_facenet_eval.py baseline pilot_eps5_seed0`

In [ ]:
W6 = '/content/drive/MyDrive/faceguard/week6'
pilot_inputs = {n: f'{W6}/eps5_n4_seed0/{n}_eps5.png' for n in names}   # baseline과 입력 폴더·파일 이름만 다름
pilot = run_gen('pilot_eps5_seed0', pilot_inputs)
show('pilot_eps5_seed0', pilot_inputs)

### 4. Ensemble: SimSwap ArcFace + FaceID ArcFace를 함께 속인 보호 사진
- **입력:** `week6/ens_eps5_n4_seed0/person_a_k_eps5.png` (`simswap_robustness.ipynb`의 ensemble 공격 셀에서 만듦)
- 설정은 pilot과 같음 (eps 5/255, 변형 4개, steps 30, 노이즈 seed 0). 다른 점은 공격 loss에 FaceID의 ArcFace를 더한 것뿐
- 사전 확인 (입력 단계 FaceID ArcFace 유사도): Week 6 보호 평균 0.642 → ensemble 0.015
- `PROMPT`, `SEEDS`는 baseline·pilot과 같다 → 입력의 노이즈만 다른 비교
- FaceNet 채점: `experiments/week7_facenet_eval.py baseline pilot_eps5_seed0 ens_eps5_seed0`


In [ ]:
ens_inputs = {n: f'{W6}/ens_eps5_n4_seed0/{n}_eps5.png' for n in names}   # pilot과 입력 폴더만 다름
ens = run_gen('ens_eps5_seed0', ens_inputs)
show('ens_eps5_seed0', ens_inputs)


### 5. Ensemble 반복 (노이즈 seed 1, 2)
- 먼저 `simswap_robustness.ipynb`의 `[ensemble 반복]` 셀로 `week6/ens_eps5_n4_seed1`, `seed2` 보호 사진을 만든다
- 생성 seed(`SEEDS`)와 프롬프트는 그대로 → 노이즈 seed만 다른 비교
- 끝나면 Drive `week7/ens_eps5_seed1/`, `ens_eps5_seed2/`를 Mac `data/week7/`로 내려받는다
- FaceNet 채점: `experiments/week7_facenet_eval.py baseline pilot_eps5_seed0 ens_eps5_seed0 ens_eps5_seed1 ens_eps5_seed2`


In [ ]:
W6 = '/content/drive/MyDrive/faceguard/week6'
for s in [1, 2]:
    inputs = {n: f'{W6}/ens_eps5_n4_seed{s}/{n}_eps5.png' for n in names}
    run_gen(f'ens_eps5_seed{s}', inputs)
